In [0]:
# %restart_python

In [0]:
# %pip install pypdf pypdfium2 pillow openai --quiet
# dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
for wh in w.warehouses.list():
    print(wh.id, "|", wh.name, "|", wh.state)

37a0db839bacb60b | Serverless Starter Warehouse | State.STOPPED


In [0]:
import io, json, base64
from pypdf import PdfReader
import pypdfium2 as pdfium
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.sql import StatementParameterListItem

TABLE        = "workspace.default.pdf_results"
VOLUME_PATH  = "/Volumes/workspace/default/raw_pdfs"
LLM_ENDPOINT = "databricks-claude-sonnet-5"
WAREHOUSE_ID = "37a0db839bacb60b"
MAX_PAGES    = 5

w   = WorkspaceClient()
llm = w.serving_endpoints.get_open_ai_client()


def save_pdf_copy(job_id, file_name, pdf_bytes):
    path = f"{VOLUME_PATH}/{job_id}_{file_name}"
    w.files.upload(path, io.BytesIO(pdf_bytes), overwrite=True)
    return path


def ocr_with_vision(pdf_bytes):
    pdf = pdfium.PdfDocument(pdf_bytes)
    content = [{"type": "text",
                "text": "Transcribe all text on these pages exactly. Return only the text."}]
    for i in range(len(pdf)):
        img = pdf[i].render(scale=2).to_pil()
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        content.append({"type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{b64}"}})
    resp = llm.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "user", "content": content}],
        max_tokens=4000)
    return resp.choices[0].message.content.strip()


def extract_text(pdf_bytes):
    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages = len(reader.pages)
    if pages > MAX_PAGES:
        raise ValueError(f"PDF has {pages} pages; limit is {MAX_PAGES}")
    text = "\n".join((p.extract_text() or "") for p in reader.pages).strip()
    if len(text) >= 50:
        return text, pages, "text-layer"
    return ocr_with_vision(pdf_bytes), pages, "ocr-vision"


ENRICH_PROMPT = """You are a document analysis agent. Read the document and return ONLY a JSON object with these keys:
"document_type": a short label such as invoice, contract, resume, report or letter,
"key_points": a list of 3 to 6 short strings,
"entities": an object with lists named "people", "organizations", "dates", "amounts",
"summary": a clear summary in 4 to 6 sentences.
Return no markdown and no extra text.

DOCUMENT:
{text}"""


def enrich_and_summarize(text):
    resp = llm.chat.completions.create(
        model=LLM_ENDPOINT,
        messages=[{"role": "user", "content": ENRICH_PROMPT.format(text=text[:30000])}],
        max_tokens=1500)
    raw = resp.choices[0].message.content.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return json.loads(raw)


def save_result(row):
    sql = f"""
    INSERT INTO {TABLE}
      (job_id, file_name, page_count, extracted_text, document_type,
       key_points, entities, summary, status, processed_by, created_at)
    VALUES
      (:job_id, :file_name, CAST(:page_count AS INT), :extracted_text, :document_type,
       :key_points, :entities, :summary, :status, :processed_by, current_timestamp())"""
    params = [StatementParameterListItem(name=k, value=str(v)) for k, v in row.items()]
    res = w.statement_execution.execute_statement(
        warehouse_id=WAREHOUSE_ID, statement=sql,
        parameters=params, wait_timeout="30s")
    if res.status.state.value != "SUCCEEDED":
        raise RuntimeError(f"Insert failed: {res.status.error}")


def process_pdf(job_id, file_name, pdf_bytes):
    row = {"job_id": job_id, "file_name": file_name, "page_count": 0,
           "extracted_text": "", "document_type": "", "key_points": "[]",
           "entities": "{}", "summary": "", "status": "failed",
           "processed_by": "databricks-pdf-agent"}
    try:
        volume_path = save_pdf_copy(job_id, file_name, pdf_bytes)
        text, pages, method = extract_text(pdf_bytes)
        ai = enrich_and_summarize(text)
        row.update({"page_count": pages, "extracted_text": text,
                    "document_type": ai.get("document_type", ""),
                    "key_points": json.dumps(ai.get("key_points", [])),
                    "entities": json.dumps(ai.get("entities", {})),
                    "summary": ai.get("summary", ""), "status": "done"})
        save_result(row)
        return {"job_id": job_id, "status": "done",
                "document_type": row["document_type"],
                "key_points": ai.get("key_points", []),
                "entities": ai.get("entities", {}),
                "summary": row["summary"], "page_count": pages,
                "extraction_method": method, "volume_path": volume_path,
                "stored_in": TABLE, "processed_by": "databricks-pdf-agent"}
    except Exception as e:
        row["summary"] = f"Error: {e}"
        try:
            save_result(row)
        except Exception:
            pass
        raise

print("PDF agent loaded")

PDF agent loaded


In [0]:
with open(f"{VOLUME_PATH}/sample.pdf", "rb") as f:
    data = f.read()

result = process_pdf("test-001", "sample.pdf", data)
print(json.dumps(result, indent=2))

{
  "job_id": "test-001",
  "status": "done",
  "document_type": "resume",
  "key_points": [
    "AI Engineer with expertise in RAG systems, LangGraph agentic workflows, and computer vision pipelines",
    "Currently employed at Bilvantis Technologies as Programmer Analyst - Artificial Intelligence for 3 years",
    "Built JiraCopilot, a LangGraph-based agentic VS Code extension with human-in-the-loop checkpoints",
    "Developed RADEX, an enterprise multi-user RAG platform with RBAC and SSO",
    "Created real-time computer vision analytics for vehicle detection using YOLO, OpenCV, and Gemini",
    "Holds a Bachelor of Technology in Computer Science Engineering from Mahindra University"
  ],
  "entities": {
    "people": [
      "Krish Hindocha"
    ],
    "organizations": [
      "Bilvantis Technologies",
      "Mahindra University",
      "OpenAI",
      "Google Gemini",
      "Neo4j",
      "Milvus",
      "MongoDB",
      "PostgreSQL"
    ],
    "dates": [],
    "amounts": [
     

In [0]:
%sql
SELECT job_id, file_name, page_count, document_type, status, processed_by, created_at, summary
FROM workspace.default.pdf_results
ORDER BY created_at DESC

job_id,file_name,page_count,document_type,status,processed_by,created_at,summary
1996f037bdce,sample_invoice.pdf,2,invoice,done,databricks-pdf-agent,2026-09-25T12:06:05.107Z,"This tax invoice, numbered INV-2026-0917, is issued by Fabrikam Logistics FZE to Contoso Retail LLC for logistics services provided in August 2026. The invoice covers cold-chain warehousing, last-mile deliveries, and customs clearance filings, totaling AED 102,375 after 5% VAT. Payment is due within 30 days of the invoice date, with late payments subject to a 1.5% monthly charge. The document also reports delivery performance for the period, noting 98.2% on-time delivery, which exceeded the 97% contractual target, and mentions two minor temperature excursions in cold storage that were quickly resolved without product loss. Contacts for account management and dispute resolution are provided, with disputes required to be raised within 10 business days of receipt."
267ea86a9ec5,sample_invoice.pdf,2,invoice,done,databricks-pdf-agent,2026-09-25T11:44:46.611Z,"This is a tax invoice (INV-2026-0917) issued by Fabrikam Logistics FZE to Contoso Retail LLC for logistics services rendered in August 2026, including cold-chain warehousing, last-mile deliveries, and customs clearance filings. The total amount due is AED 102,375, including 5% VAT, payable within 30 days of the invoice date, with late payments subject to a 1.5% monthly charge. The document also reports delivery performance metrics, noting that 98.2% of deliveries were completed on time, surpassing the contractual target of 97%. Two temperature excursions in cold storage on 12 August 2026 were resolved quickly without product loss. The invoice specifies a 10-business-day window for disputes and lists Omar Haddad as the account manager and Sara Al Mansoori as the client contact."
f70e2ece5ed1,01_Java_Job_Description.pdf,2,job posting,done,databricks-pdf-agent,2026-09-25T11:34:10.333Z,"This document is a job posting for a Senior Java Developer position requiring at least 5 years of professional Java development experience. The role involves designing and maintaining enterprise-level applications and microservices using Spring Boot, Spring Cloud, and related frameworks. Candidates should have strong knowledge of design patterns, database technologies, ORM frameworks, and concurrency management. Preferred qualifications include experience with cloud platforms, containerization tools, and messaging systems like Kafka or RabbitMQ. The role also involves mentoring junior developers, implementing CI/CD pipelines, and collaborating with DevOps teams. Compensation includes a competitive salary, health insurance, retirement benefits, and performance bonuses."
b9449b58ff8d,sample_invoice.pdf,0,,failed,databricks-pdf-agent,2026-09-25T11:29:51.966Z,Error: 'list' object has no attribute 'strip'
5ffe6d37b4cb,sample.pdf,0,,failed,databricks-pdf-agent,2026-09-25T11:28:25.468Z,Error: 'list' object has no attribute 'strip'
f2ed7f647cdf,01_Java_Job_Description.pdf,2,job posting,done,databricks-pdf-agent,2026-09-25T11:01:37.514Z,"This document is a job posting for a Senior Java Developer position requiring at least 5 years of professional experience. The role involves designing and maintaining scalable microservices and enterprise applications using Java, Spring Boot, and related frameworks. Candidates should have strong knowledge of design patterns, database technologies, ORM frameworks, and multithreading, along with experience in CI/CD pipelines and testing tools like JUnit and Mockito. Preferred qualifications include cloud platform experience, containerization with Docker/Kubernetes, and familiarity with message brokers and NoSQL databases. The position offers competitive compensation, health insurance, retirement benefits, and performance-based bonuses. Overall, the posting outlines both technical responsibilities and required/preferred skills for a senior-level software engineering role."
681ea7b38e4a,sample.pdf,1,sample